# Tier 2: DQN on LunarLander-v3 (discrete)

1. **Vanilla DQN** (experience replay + target network) on a harder discrete-action problem.
2. **Hyperparameter study**: learning rate and target-network update frequency (one at a time, 3 values each).
3. **Ablation**: DQN without target network, without replay buffer, and without both.

## 0. Setup

Requires `pip install "gymnasium[box2d]" torch matplotlib pillow joblib`.

In [1]:
import copy
import json
import os
import platform
import random
import time
from pathlib import Path

import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import Image as ShowImage, Markdown, display
from PIL import Image

torch.set_num_threads(1)   # the network is small: 1 thread is faster


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


HARDWARE = f"{platform.processor() or platform.machine()}, {os.cpu_count()} CPUs, {platform.system()}"
print("python", platform.python_version(), "| torch", torch.__version__,
      "| gymnasium", gym.__version__, "| numpy", np.__version__)
print("hardware:", HARDWARE)

python 3.11.9 | torch 2.10.0+cpu | gymnasium 1.2.2 | numpy 2.2.6
hardware: Intel64 Family 6 Model 183 Stepping 1, GenuineIntel, 32 CPUs, Windows


### Configuration

| | Value |
| --- | --- |
| Step budget | 500,000 environment steps per run (same for every configuration) |
| Seeds | 3 per configuration (0, 1, 2) |
| Evaluation | 10 greedy episodes (ε = 0) every 10,000 steps; 100 greedy episodes at the end |
| Success threshold | mean return ≥ 200 over 100 episodes |
| γ | 0.99 |
| Default DQN | MLP 8 → 256 → 256 → 4, Adam lr 5e-4, buffer 100k, batch 128, warmup 10k, one update every 4 steps, target update every C = 1,000 steps, ε 1.0 → 0.05 in 100k steps, Huber loss, gradient-norm clip 10 |

In [2]:
ENV_ID = "LunarLander-v3"
SUCCESS_THRESHOLD = 200   # solved: mean return >= 200 over 100 episodes
GAMMA = 0.99

STEP_BUDGET = 500_000
SEEDS = [0, 1, 2]
EVAL_EVERY = 10_000
EVAL_EPISODES = 10     # episodes in each evaluation during training
FINAL_EPISODES = 100   # episodes in the final evaluation

# runs trained at the same time (each run uses 1 CPU thread). Use 1 for fully sequential training.
N_JOBS = max(1, min(12, (os.cpu_count() or 2) - 1))

RESULTS_DIR = Path("results/tier2")
LOG_DIR = RESULTS_DIR / "logs"       # metrics of each run (JSON)
MODEL_DIR = RESULTS_DIR / "models"   # network weights
FIG_DIR = RESULTS_DIR / "figures"    # plots and GIFs
for folder in [LOG_DIR, MODEL_DIR, FIG_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print(f"steps per run: {STEP_BUDGET:,} | seeds: {SEEDS} | evaluate every {EVAL_EVERY:,} steps | parallel runs: {N_JOBS}")

steps per run: 500,000 | seeds: [0, 1, 2] | evaluate every 10,000 steps | parallel runs: 12


## 1. Pertinence

| Element | LunarLander-v3 (discrete) |
| --- | --- |
| State | 8 continuous: x, y, vx, vy, angle, angular velocity, left leg contact, right leg contact |
| Actions | 4 discrete: nothing (0), left engine (1), main engine (2), right engine (3) |
| Reward | shaping for moving towards the pad and slowing down, −0.3 per frame of main engine, −0.03 per frame of side engine, +10 per leg contact, +100 landing / −100 crash |
| End | `terminated`: crash, out of the screen, or lander at rest. `truncated`: 1,000 steps |

- **DQN**: works, because the actions are discrete.
- **Tabular**: not practical. 8 continuous dimensions with only 6 bins each would already give $6^8 \approx 1.7$ million states, and the landing needs fine control of position and speed.
- Harder than CartPole: sparse terminal reward (±100), longer episodes, and a "hovering" local optimum (staying in the air avoids the crash penalty but never collects the landing bonus).

In [3]:
env = gym.make(ENV_ID)
print("observation space:", env.observation_space)
print("action space     :", env.action_space)
print("max episode steps:", env.spec.max_episode_steps)

# random policy as a baseline
returns, lengths = [], []
env.action_space.seed(0)
for k in range(100):
    state, _ = env.reset(seed=k)
    done, total, n = False, 0.0, 0
    while not done:
        state, reward, terminated, truncated, _ = env.step(env.action_space.sample())
        total += reward
        n += 1
        done = terminated or truncated
    returns.append(total)
    lengths.append(n)
env.close()
RANDOM_RETURN = float(np.mean(returns))
print(f"random policy (100 episodes): return {np.mean(returns):.1f} ± {np.std(returns):.1f}, "
      f"length {np.mean(lengths):.0f} ± {np.std(lengths):.0f} steps")

DependencyNotInstalled: Box2D is not installed, you can install it by run `pip install swig` followed by `pip install "gymnasium[box2d]"`

The random policy crashes quickly with a strongly negative return; it is shown as a reference line in the plots.

## 2. Evaluation and logging

Same protocol as Tier 1:

- `epsilon_by_step`: linear decay of ε, then fixed.
- `evaluate`: mean return of the **greedy** policy (ε = 0) in a separate environment with fixed evaluation seeds.
- Training time does **not** include evaluation time.

In [ ]:
def epsilon_by_step(i, eps_start, eps_end, eps_decay):
    fraction = min(1.0, i / eps_decay)
    return eps_start + fraction * (eps_end - eps_start)


EVAL_SEED = 100_000    # evaluation seeds: different from the training seeds
FINAL_SEED = 200_000   # seeds of the final 100-episode evaluation


def evaluate(policy, num_episodes, first_seed, return_all=False):
    # policy: function that takes a state and returns an action
    env = gym.make(ENV_ID)
    returns = []
    for k in range(num_episodes):
        state, _ = env.reset(seed=first_seed + k)
        done = False
        total_reward = 0.0
        while not done:
            state, reward, terminated, truncated, _ = env.step(policy(state))
            total_reward += reward
            done = terminated or truncated
        returns.append(total_reward)
    env.close()
    if return_all:
        return returns
    return float(np.mean(returns))


def new_log(method, seed, config):
    return {
        "method": method, "seed": seed, "config": config, "hardware": HARDWARE,
        "eval_steps": [],          # step of each evaluation
        "eval_returns": [],        # greedy return of that evaluation
        "eval_time": [],           # training seconds until that evaluation
        "episode_end_steps": [],   # step where each training episode ended
        "episode_returns": [],     # return of each training episode
    }


def add_eval(log, step, policy, train_seconds):
    log["eval_steps"].append(step)
    log["eval_returns"].append(evaluate(policy, EVAL_EPISODES, EVAL_SEED))
    log["eval_time"].append(train_seconds)


def steps_to_threshold(log):
    # First step where the evaluation reached the threshold, or None if it never did
    for step, ret in zip(log["eval_steps"], log["eval_returns"]):
        if ret >= SUCCESS_THRESHOLD:
            return step
    return None

## 3. DQN

- **`QNetwork`**: MLP 8 → `hidden` → `hidden` → 4 (one Q-value per action).
- **`ReplayBuffer`**: circular memory; random mini-batches break the correlation between consecutive steps and reuse each transition many times.
- **Target network** θ⁻: copy of the network, updated every C environment steps.
- **Loss**: Huber (smooth L1) between $Q(s,a;\theta)$ and $y = r + \gamma\,(1 - terminated)\,\max_{a'} Q(s',a';\theta^-)$. Huber is used instead of the MSE of Tier 1 because LunarLander rewards are much larger (±100), so squared errors would give very large gradients; the gradient norm is also clipped to 10.

At `truncated` (1,000 steps) the state is not terminal, so we still bootstrap (only `terminated` is stored as `done`).

In [ ]:
class QNetwork(nn.Module):
    def __init__(self, state_dim, n_actions, hidden=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, n_actions),
        )

    def forward(self, x):
        return self.net(x)          # shape: (batch, n_actions)


class ReplayBuffer:
    def __init__(self, capacity, state_dim):
        self.N = capacity
        self.c = 0                                   # transitions ever added
        self.s    = np.zeros((capacity, state_dim), dtype=np.float32)
        self.a    = np.zeros(capacity, dtype=np.int64)
        self.r    = np.zeros(capacity, dtype=np.float32)
        self.s2   = np.zeros((capacity, state_dim), dtype=np.float32)
        self.done = np.zeros(capacity, dtype=np.float32)

    def add(self, s, a, r, s2, done):
        i = self.c % self.N                          # position to write
        self.s[i] = s
        self.a[i] = a
        self.r[i] = r
        self.s2[i] = s2
        self.done[i] = done
        self.c += 1

    def __len__(self):
        return min(self.c, self.N)

    def sample(self, n):
        idx = np.random.randint(0, len(self), size=n)   # n random indices
        return self.s[idx], self.a[idx], self.r[idx], self.s2[idx], self.done[idx]

    def all(self):
        # every stored transition (used by the "no replay" ablation)
        n = len(self)
        return self.s[:n], self.a[:n], self.r[:n], self.s2[:n], self.done[:n]


def select_action(q_net, s, eps, n_actions):
    if random.random() < eps:
        return random.randrange(n_actions)
    with torch.no_grad():
        q_values = q_net(torch.as_tensor(s, dtype=torch.float32).unsqueeze(0))
    return int(q_values.argmax(dim=1).item())


def compute_loss(q_net, target_net, batch, gamma):
    s, a, r, s2, done = (torch.as_tensor(x) for x in batch)
    q_sa = q_net(s).gather(1, a.unsqueeze(1)).squeeze(1)          # Q(s, a; θ) of the action taken
    with torch.no_grad():
        max_next = target_net(s2).max(dim=1).values                # max_a' Q(s', a'; θ⁻)
        target = r + gamma * (1 - done) * max_next
    return F.smooth_l1_loss(q_sa, target)                         # Huber loss

### 3.1 Training loop

1. ε-greedy action → store the transition in the buffer.
2. Every `train_every` = 4 steps, after `warmup`: sample a mini-batch, compute the loss, one Adam step.
3. Every `C` steps: copy the weights to the target network.

**Ablation switches**

| Flag | Effect |
| --- | --- |
| `use_target=False` | No target network: the bootstrap target uses the **online** network θ, so the target moves at every update. |
| `use_replay=False` | No replay buffer: the buffer only keeps the last `train_every` = 4 transitions; each update uses exactly those (consecutive, correlated) transitions, so every transition is used **once** and then thrown away. Same number of gradient updates as vanilla DQN. |

In [ ]:
def train_dqn(name, seed, num_steps=STEP_BUDGET, lr=5e-4, C=1_000, capacity=100_000, batch=128,
              warmup=10_000, train_every=4, hidden=256, gamma=GAMMA, eps_start=1.0, eps_end=0.05,
              eps_decay=100_000, grad_clip=10.0, use_target=True, use_replay=True):
    if not use_replay:
        # online learning: the mini-batch is the transitions just collected, used once
        capacity, batch, warmup = train_every, train_every, 0
    config = {"num_steps": num_steps, "lr": lr, "C": C, "capacity": capacity, "batch": batch,
              "warmup": warmup, "train_every": train_every, "hidden": hidden, "gamma": gamma,
              "eps_start": eps_start, "eps_end": eps_end, "eps_decay": eps_decay, "grad_clip": grad_clip,
              "use_target": use_target, "use_replay": use_replay}
    torch.set_num_threads(1)
    set_seed(seed)
    env = gym.make(ENV_ID)
    env.action_space.seed(seed)
    state_dim = env.observation_space.shape[0]
    n_actions = env.action_space.n

    q_net = QNetwork(state_dim, n_actions, hidden)
    if use_target:
        target_net = copy.deepcopy(q_net)                  # θ⁻ ← θ
    else:
        target_net = q_net                                 # ablation: target computed with θ itself
    opt = torch.optim.Adam(q_net.parameters(), lr=lr)
    buf = ReplayBuffer(capacity, state_dim)

    def greedy_policy(state):
        return select_action(q_net, state, 0.0, n_actions)

    log = new_log("dqn", seed, config)
    start = time.perf_counter()
    eval_seconds = 0.0
    add_eval(log, 0, greedy_policy, 0.0)
    eval_seconds += time.perf_counter() - start

    s, _ = env.reset(seed=seed)
    episode_return = 0.0
    for i in range(1, num_steps + 1):
        # 1) act and store the transition
        eps = epsilon_by_step(i, eps_start, eps_end, eps_decay)
        a = select_action(q_net, s, eps, n_actions)
        s2, r, terminated, truncated, _ = env.step(a)
        buf.add(s, a, r, s2, float(terminated))
        s = s2
        episode_return += r
        if terminated or truncated:
            log["episode_end_steps"].append(i)
            log["episode_returns"].append(episode_return)
            s, _ = env.reset()
            episode_return = 0.0

        # 2) learn from a mini-batch
        if i % train_every == 0 and len(buf) >= max(warmup, batch):
            mini_batch = buf.sample(batch) if use_replay else buf.all()
            loss = compute_loss(q_net, target_net, mini_batch, gamma)
            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(q_net.parameters(), grad_clip)
            opt.step()

        # 3) update the target network
        if use_target and i % C == 0:
            target_net.load_state_dict(q_net.state_dict())

        if i % EVAL_EVERY == 0:
            t0 = time.perf_counter()
            add_eval(log, i, greedy_policy, t0 - start - eval_seconds)
            eval_seconds += time.perf_counter() - t0
    env.close()

    log["train_time"] = time.perf_counter() - start - eval_seconds
    final = evaluate(greedy_policy, FINAL_EPISODES, FINAL_SEED, return_all=True)
    log["final_returns"] = final
    log["final_return"] = float(np.mean(final))
    torch.save(q_net.state_dict(), MODEL_DIR / f"{name}.pt")
    return log

## 4. Experiments

All runs use the default configuration except for **one** changed parameter (one at a time).

| Group | Configurations (× 3 seeds) |
| --- | --- |
| Default | vanilla DQN (lr 5e-4, C = 1,000) |
| Learning rate | lr ∈ {1e-4, **5e-4**, 1e-3} |
| Target update frequency | C ∈ {250, **1,000**, 5,000} steps |
| Ablation | no target network · no replay buffer · no target and no replay |

Bold = default (shared with the "Default" runs, not trained again). 8 configurations × 3 seeds = 24 runs.

Each run saves its log in `results/tier2/logs/`. Runs that are already saved are loaded, not trained again, so the notebook can be stopped and restarted. Runs are trained in parallel with `joblib` (`N_JOBS` processes); set `N_JOBS = 1` to train sequentially.

In [ ]:
# label -> (file prefix, parameters that differ from the default)
EXPERIMENTS = {
    "DQN (default)":       ("dqn_default", {}),
    "lr = 1e-4":           ("dqn_lr1e-4", {"lr": 1e-4}),
    "lr = 1e-3":           ("dqn_lr1e-3", {"lr": 1e-3}),
    "C = 250":             ("dqn_C250", {"C": 250}),
    "C = 5000":            ("dqn_C5000", {"C": 5_000}),
    "No target network":   ("dqn_no_target", {"use_target": False}),
    "No replay buffer":    ("dqn_no_replay", {"use_replay": False}),
    "No target, no replay": ("dqn_no_target_no_replay", {"use_target": False, "use_replay": False}),
}


def run_job(name, params):
    torch.set_num_threads(1)
    log = train_dqn(name, **params)
    log["n_jobs"] = N_JOBS
    with open(LOG_DIR / f"{name}.json", "w") as f:
        json.dump(log, f)
    return name, log["final_return"], log["train_time"]


def run_all(jobs):
    pending = [job for job in jobs if not (LOG_DIR / f"{job[0]}.json").exists()]
    print(f"{len(jobs) - len(pending)} runs already saved, {len(pending)} to train", flush=True)
    if N_JOBS > 1 and len(pending) > 1:
        from joblib import Parallel, delayed
        results = Parallel(n_jobs=N_JOBS, return_as="generator_unordered")(
            delayed(run_job)(name, params) for name, params in pending)
    else:
        results = (run_job(name, params) for name, params in pending)
    for name, final_return, train_time in results:
        print(f"done {name:34s} final return = {final_return:7.1f}   time = {train_time:7.1f} s", flush=True)

    logs = {}
    for name, _ in jobs:
        with open(LOG_DIR / f"{name}.json") as f:
            logs[name] = json.load(f)
    return logs


jobs = []
for seed in SEEDS:
    for label, (prefix, overrides) in EXPERIMENTS.items():
        jobs.append((f"{prefix}_s{seed}", {"seed": seed, **overrides}))

t0 = time.perf_counter()
LOGS = run_all(jobs)
print(f"total time: {time.perf_counter() - t0:.0f} s")

## 5. Results

Mean over the 3 seeds (line) ± one standard deviation (band).
- **Evaluation curve:** greedy return (10 episodes every 10,000 steps).
- **Training curve:** moving average of 20 training episodes (ε-greedy), interpolated on a common step grid.
- Dashed line: success threshold (200). Dotted line: random policy.

In [ ]:
GROUPS = {label: [LOGS[f"{prefix}_s{seed}"] for seed in SEEDS] for label, (prefix, _) in EXPERIMENTS.items()}


def eval_curve(logs):
    x = np.array(logs[0]["eval_steps"])
    Y = np.array([log["eval_returns"] for log in logs])   # one row per seed
    return x, Y


def train_curve(logs, window=20):
    x = np.linspace(0, STEP_BUDGET, 200)
    Y = []
    for log in logs:
        rets = np.array(log["episode_returns"])
        smooth = np.convolve(rets, np.ones(window) / window, mode="valid")
        steps = np.array(log["episode_end_steps"])[window - 1:]
        Y.append(np.interp(x, steps, smooth))
    return x, np.array(Y)


def plot_mean_std(ax, x, Y, label, **kwargs):
    mean = Y.mean(axis=0)
    std = Y.std(axis=0)
    line = ax.plot(x, mean, label=label, **kwargs)[0]
    ax.fill_between(x, mean - std, mean + std, color=line.get_color(), alpha=0.2)


def decorate(ax, ylabel, title):
    ax.axhline(SUCCESS_THRESHOLD, color="black", linestyle="--", linewidth=1, label=f"threshold {SUCCESS_THRESHOLD}")
    ax.axhline(RANDOM_RETURN, color="gray", linestyle=":", linewidth=1, label="random policy")
    ax.set_ylabel(ylabel)
    ax.set_title(title, fontsize=10)
    ax.legend(fontsize=8)


def plot_groups(names, title, filename):
    # left: evaluation (greedy) | right: training (ε-greedy)
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
    for name in names:
        plot_mean_std(axes[0], *eval_curve(GROUPS[name]), name)
        plot_mean_std(axes[1], *train_curve(GROUPS[name]), name)
    decorate(axes[0], "evaluation return (greedy)", f"{title}: evaluation — {len(SEEDS)} seeds, mean ± sd")
    decorate(axes[1], "training return (moving avg. 20)", f"{title}: training — {len(SEEDS)} seeds, mean ± sd")
    for ax in axes:
        ax.set_xlabel("environment steps")
        ax.set_ylim(bottom=-400)
    fig.tight_layout()
    fig.savefig(FIG_DIR / filename, dpi=150)
    plt.show()

### 5.1 Default DQN

Evaluation and training curves; thin lines are the individual seeds.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
x, Y = eval_curve(GROUPS["DQN (default)"])
xt, Yt = train_curve(GROUPS["DQN (default)"])
for k, seed in enumerate(SEEDS):
    axes[0].plot(x, Y[k], linewidth=0.7, alpha=0.6, label=f"seed {seed}")
    axes[1].plot(xt, Yt[k], linewidth=0.7, alpha=0.6, label=f"seed {seed}")
plot_mean_std(axes[0], x, Y, "mean ± sd", color="black")
plot_mean_std(axes[1], xt, Yt, "mean ± sd", color="black")
decorate(axes[0], "evaluation return (greedy)", "Default DQN: evaluation")
decorate(axes[1], "training return (moving avg. 20)", "Default DQN: training")
for ax in axes:
    ax.set_xlabel("environment steps")
    ax.set_ylim(bottom=-400)
fig.tight_layout()
fig.savefig(FIG_DIR / "default_dqn.png", dpi=150)
plt.show()

### 5.2 Hyperparameter study: learning rate

Only the learning rate changes; everything else is the default.

In [ ]:
LR_NAMES = ["lr = 1e-4", "DQN (default)", "lr = 1e-3"]
plot_groups(LR_NAMES, "Learning rate (default 5e-4)", "study_lr.png")

### 5.3 Hyperparameter study: target-network update frequency C

Only C changes; everything else is the default.

In [ ]:
C_NAMES = ["C = 250", "DQN (default)", "C = 5000"]
plot_groups(C_NAMES, "Target update every C steps (default 1,000)", "study_C.png")

### 5.4 Ablation

Vanilla DQN vs DQN without target network, without replay buffer, and without both. The last plot uses training time (log scale) on the X axis.

In [ ]:
ABLATION_NAMES = ["DQN (default)", "No target network", "No replay buffer", "No target, no replay"]
plot_groups(ABLATION_NAMES, "Ablation", "ablation.png")

fig, ax = plt.subplots(figsize=(8, 4.5))
for name in ABLATION_NAMES:
    logs = GROUPS[name]
    x = np.mean([log["eval_time"] for log in logs], axis=0)
    Y = eval_curve(logs)[1]
    plot_mean_std(ax, x[1:], Y[:, 1:], name)   # skip point 0 (time 0 does not exist on a log scale)
ax.axhline(SUCCESS_THRESHOLD, color="black", linestyle="--", linewidth=1)
ax.set_xscale("log")
ax.set_xlabel("training time [s] (log scale)")
ax.set_ylabel("evaluation return (greedy)")
ax.set_title("Ablation: return vs training time")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIG_DIR / "ablation_walltime.png", dpi=150)
plt.show()

### 5.5 Final return of every configuration

Greedy return over the final 100 episodes: bar = mean over seeds, error bar = sd across seeds, dots = individual seeds.

In [ ]:
names = list(GROUPS)
finals = np.array([[log["final_return"] for log in GROUPS[n]] for n in names])   # (configs, seeds)
colors = ["tab:blue"] + ["tab:orange"] * 2 + ["tab:green"] * 2 + ["tab:red"] * 3

fig, ax = plt.subplots(figsize=(10, 4.5))
pos = np.arange(len(names))
ax.bar(pos, finals.mean(axis=1), yerr=finals.std(axis=1), color=colors, alpha=0.7, capsize=4)
for k in range(len(SEEDS)):
    ax.scatter(pos, finals[:, k], color="black", s=12, zorder=3)
ax.axhline(SUCCESS_THRESHOLD, color="black", linestyle="--", linewidth=1)
ax.axhline(0, color="gray", linewidth=0.8)
ax.set_xticks(pos)
ax.set_xticklabels(names, rotation=25, ha="right")
ax.set_ylabel("final return (100 greedy episodes)")
ax.set_title("Default (blue) · learning rate (orange) · target update C (green) · ablation (red)")
fig.tight_layout()
fig.savefig(FIG_DIR / "final_returns.png", dpi=150)
plt.show()

### 5.6 Summary table

- **Final eval**: last 10-episode evaluation (step 500k).
- **Final 100 ep.**: greedy return over 100 episodes, mean ± sd **across seeds**; **Solved** = seeds with ≥ 200.
- **Best eval**: best 10-episode evaluation during training (shows instability when much higher than the final value).
- **Steps to threshold**: first evaluation ≥ 200 (median over the seeds that reach it), or "not reached".
- **Time**: training time per run, without evaluation.

In [ ]:
lines = [
    "| Configuration | Final eval | Final 100 ep. | Solved | Best eval | Steps to threshold | Reached | Time [s] |",
    "| --- | --- | --- | --- | --- | --- | --- | --- |",
]
for name, logs in GROUPS.items():
    last_eval = np.array([log["eval_returns"][-1] for log in logs])
    final100 = np.array([log["final_return"] for log in logs])
    best_eval = np.array([max(log["eval_returns"]) for log in logs])
    times = np.array([log["train_time"] for log in logs])
    solved = int(np.sum(final100 >= SUCCESS_THRESHOLD))

    hits = []
    for log in logs:
        step = steps_to_threshold(log)
        if step is not None:
            hits.append(step)
    if len(hits) > 0:
        hit_text = f"{int(np.median(hits)):,}"
    else:
        hit_text = "not reached"

    lines.append(f"| {name} | {last_eval.mean():.1f} ± {last_eval.std():.1f} | {final100.mean():.1f} ± {final100.std():.1f} "
                 f"| {solved}/{len(logs)} | {best_eval.mean():.1f} ± {best_eval.std():.1f} | {hit_text} | {len(hits)}/{len(logs)} "
                 f"| {times.mean():.0f} ± {times.std():.0f} |")

table = "\n".join(lines)
first_log = LOGS[f"dqn_default_s{SEEDS[0]}"]
train_hardware = first_log["hardware"]   # machine where the runs were trained
parallel = first_log.get("n_jobs", 1)
header = (f"Hardware: {train_hardware} | runs trained in parallel: {parallel} (1 CPU thread each) | "
          f"budget {STEP_BUDGET:,} steps | seeds {SEEDS}")
(RESULTS_DIR / "summary.md").write_text(f"{header}\n\n{table}\n")
display(Markdown(table))
print(header)

## 6. GIF of the trained agent

The run with the highest final return of the default DQN, and for contrast the best run of the "no target, no replay" ablation.

In [ ]:
def load_dqn_policy(name):
    env = gym.make(ENV_ID)
    state_dim, n_actions = env.observation_space.shape[0], env.action_space.n
    env.close()
    hidden = LOGS[name]["config"]["hidden"]
    q_net = QNetwork(state_dim, n_actions, hidden)
    q_net.load_state_dict(torch.load(MODEL_DIR / f"{name}.pt"))

    def policy(state):
        return select_action(q_net, state, 0.0, n_actions)
    return policy


def record_gif(policy, path):
    env = gym.make(ENV_ID, render_mode="rgb_array")
    state, _ = env.reset(seed=FINAL_SEED)
    frames = [env.render()]
    done = False
    total_reward = 0.0
    while not done:
        state, reward, terminated, truncated, _ = env.step(policy(state))
        frames.append(env.render())
        total_reward += reward
        done = terminated or truncated
    env.close()
    images = [Image.fromarray(frame).resize((360, 240)) for frame in frames[::2]]   # 1 of every 2 frames
    images[0].save(path, save_all=True, append_images=images[1:], duration=40, loop=0)
    return total_reward


for label in ["DQN (default)", "No target, no replay"]:
    prefix = EXPERIMENTS[label][0]
    runs = [f"{prefix}_s{seed}" for seed in SEEDS]
    best = max(runs, key=lambda name: LOGS[name]["final_return"])   # highest return over the 100 final episodes
    path = FIG_DIR / f"{prefix}_best.gif"
    ret = record_gif(load_dqn_policy(best), path)
    print(f"{label}: {best}, return of the recorded episode = {ret:.0f}")
    display(ShowImage(filename=str(path)))